# Jour 2 · Premier réseau Dense sur des images

## Objectifs

- comprendre une couche Dense et une sortie softmax
- entraîner et valider un petit classifieur d’images
- analyser les erreurs au-delà de l’accuracy


## Plusieurs neurones forment une couche

Une couche **Dense**, ou entièrement connectée, contient plusieurs neurones. Chaque neurone reçoit toutes les sorties de la couche précédente. Une succession de couches transforme progressivement le vecteur d'entrée en scores de classes.


![Réseau Dense reliant plusieurs couches](../ressources/illustrations/jour_02/02_reseau_dense.png)

*Chaque trait représente un poids distinct appris pendant l’entraînement.*


## Aplatir une image

Notre image 8 × 8 contient 64 pixels. Une couche `Flatten` range ces pixels dans un vecteur de 64 nombres. Le réseau Dense peut alors les combiner, mais il perd l'organisation spatiale explicite : deux pixels voisins dans l'image ne bénéficient pas d'un traitement spécial.

La dernière couche possède dix neurones, un par chiffre. **Softmax** transforme leurs scores en dix valeurs positives dont la somme vaut environ 1. Nous les lirons comme des confiances relatives entre les classes.


![Flatten convertit une grille de pixels en une liste sans modifier les valeurs](../ressources/illustrations/jour_02/02_flatten_numerique.svg)

La **couche d'entrée** décrit la forme attendue. `Flatten` réorganise les valeurs et n'apprend aucun poids. `Dense(32)` crée ensuite 32 neurones ; chacun reçoit les 64 pixels. La dernière couche possède dix neurones, un par chiffre.

Les scores avant softmax sont appelés **logits**. Softmax les transforme en valeurs positives dont la somme vaut 1. Le réseau exprime ainsi sa préférence entre les dix classes.

Petit contrôle : une image de 8 × 8 pixels donne-t-elle 16, 64 ou 128 valeurs après Flatten ? Justifiez en comptant les cases.


## Préparer les outils

Un **import** rend un outil Python disponible. `as np`, `as plt` et `as tf` lui donnent un nom court. NumPy gère les tableaux ; Matplotlib les affiche ; TensorFlow entraîne les réseaux. Les premières lignes configurent les messages et le calcul sur CPU. Elles se recopient telles quelles et ne font pas partie du modèle.

La **graine aléatoire** fixe le point de départ du hasard. `42` est un choix arbitraire : il facilite les comparaisons sans garantir des résultats exactement identiques sur toutes les machines.


In [ ]:
import os  # Réglages de l'environnement avant le chargement de TensorFlow.
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")  # Réduit les messages techniques.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")  # Utilise le CPU pour ce petit cours.

import numpy as np  # Tableaux et calculs numériques.
import matplotlib.pyplot as plt  # Graphiques et images.
import tensorflow as tf  # Réseaux et calcul automatique des gradients.

tf.keras.utils.set_random_seed(42)  # Fixe le hasard pour comparer les expériences.
try:
    # Limite le nombre de threads utilisés par TensorFlow.
    # Un thread par opération suffit ; ce réglage limite la consommation du cours.
    tf.config.threading.set_intra_op_parallelism_threads(1)
    tf.config.threading.set_inter_op_parallelism_threads(1)
except RuntimeError:
    pass  # Ce réglage ne peut plus changer si TensorFlow a déjà commencé à calculer.
print("TensorFlow :", tf.__version__)


## Reprendre les mêmes images, avec trois rôles distincts

`load_digits()` fournit de petites images de chiffres. `astype("float32")` convertit les pixels en nombres décimaux ; `int64` garde les labels sous forme d'entiers. La division par `16` place les pixels entre 0 et 1. Ce maximum vient du jeu de données, pas d'une valeur estimée sur le test.

**Train** : exemples utilisés pour corriger les poids. **Validation** : exemples utilisés pour suivre l'entraînement et choisir les réglages. **Test** : exemples réservés à la mesure finale. `train_test_split` coupe ensemble les images et leurs labels, pour conserver leur association. `stratify=labels` conserve approximativement la proportion de chaque chiffre. `random_state=42` fixe ce découpage.

Dans les noms Python, `X` désigne les entrées et `y` les réponses. Nous séparons d'abord 25 % pour le test, puis 20 % du reste pour la validation. Les proportions finales sont environ 60 % train, 15 % validation et 25 % test.


In [ ]:
from sklearn.datasets import load_digits  # Données incluses localement dans scikit-learn.
from sklearn.model_selection import train_test_split  # Découpage sans mélanger images et labels.

digits = load_digits()
images = digits.images.astype("float32") / 16.0  # Intensités de 0 à 1.
labels = digits.target.astype("int64")  # Chiffre attendu, de 0 à 9.
# Les pixels vont à l'origine de 0 à 16 ; on les ramène entre 0 et 1 pour faciliter l'apprentissage.
# stratify garde à peu près la même proportion de chaque chiffre dans les jeux.
# random_state rend le découpage reproductible.
X_train_val, X_test, y_train_val, y_test = train_test_split(
    images, labels, test_size=0.25, stratify=labels, random_state=42,
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.20, stratify=y_train_val, random_state=42,
)
print("Train :", X_train.shape, "Validation :", X_val.shape, "Test :", X_test.shape)
print("Pixels du train :", X_train.min(), "à", X_train.max())


## Construire une première version facile à lire

Le parcours est **image 8 × 8 → liste de 64 pixels → 32 neurones ReLU → dix sorties softmax**. Nous gardons une version simple pour voir le rôle de chaque couche. L'arrêt anticipé surveillera la validation. Le Dropout sera présenté ensuite comme une extension, une fois le réseau compris.

`Sequential` empile les couches dans l'ordre de la liste. `Input(shape=(8, 8))` décrit une image ; la dimension du batch est gérée automatiquement. `summary()` affiche les formes de sortie et les paramètres appris.


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(8, 8)),  # Une image de 8 lignes et 8 colonnes.
    tf.keras.layers.Flatten(),  # 64 pixels deviennent une liste ; aucun poids appris.
    tf.keras.layers.Dense(32, activation="relu"),  # 32 combinaisons apprises des pixels.
    tf.keras.layers.Dense(10, activation="softmax"),  # Une valeur par chiffre de 0 à 9.
])
model.summary()  # Formes et nombre de paramètres, avant tout entraînement.


## Choisir la loss avant de lancer l'entraînement

Le réseau donne dix scores ; la bonne réponse est un entier comme `7`. `sparse_categorical_crossentropy` compare les deux : elle pénalise un score faible pour la bonne classe. **Sparse** signifie ici que le label est un entier, au lieu d'un tableau de dix zéros avec un `1` à la bonne position. **Categorical** indique plusieurs classes exclusives : une image reçoit un seul chiffre.

La **loss** guide l'apprentissage. L'**accuracy** compte seulement la proportion de classes correctement choisies. Une prédiction correcte mais hésitante peut donc avoir la même accuracy qu'une prédiction correcte et nette, tout en ayant une loss différente.

`compile` associe le réseau, la loss et l'optimiseur. `optimizer="adam"` demande à Adam de modifier les poids grâce aux gradients. `metrics=["accuracy"]` ajoute une mesure à afficher ; elle ne remplace pas la loss.


In [ ]:
model.compile(
    optimizer="adam",  # Corrige les poids à partir des gradients.
    loss="sparse_categorical_crossentropy",  # Compare les dix scores au label entier.
    metrics=["accuracy"],  # Suit aussi la proportion de classes correctement choisies.
)


## Apprendre, puis suivre la validation

`fit` lance l'entraînement. Une **époque** est un parcours du train ; `batch_size=32` traite 32 images avant une correction des poids. Les images de `validation_data` sont utilisées pour mesurer `val_loss` et `val_accuracy`, jamais pour cette correction.

Un **callback** est une action appelée pendant l'entraînement. `EarlyStopping` observe `val_loss` : après plusieurs époques sans amélioration, il arrête le calcul. `restore_best_weights=True` recharge les meilleurs poids observés sur la validation. `epochs` fixe seulement un maximum ; `verbose=0` cache le journal détaillé.


In [ ]:
arret_anticipe = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",  # Observe l'erreur sur les images jamais utilisées pour les gradients.
    patience=4,  # Tolère quatre époques sans amélioration.
    restore_best_weights=True,  # Restaure les poids de la meilleure époque de validation.
)
history = model.fit(
    X_train, y_train,  # Les exemples qui servent à apprendre.
    validation_data=(X_val, y_val),  # Exemples qui servent seulement à contrôler.
    epochs=30, batch_size=32, callbacks=[arret_anticipe], verbose=0,
)
print("Époques réellement effectuées :", len(history.history["loss"]))


## Lire les courbes train et validation

`history.history` mémorise les valeurs mesurées après chaque époque. `loss` concerne les exemples utilisés pour apprendre ; `val_loss` concerne les exemples de validation mis de côté.

Si les deux diminuent, le modèle progresse généralement. Si la loss du train continue de baisser tandis que `val_loss` remonte durablement, le réseau commence probablement à mémoriser des particularités du train.

Le test intervient ensuite avec `model.evaluate`. Son accuracy estime la proportion d’images correctement classées sur des exemples restés hors de l’apprentissage et de l’arrêt anticipé.


![Courbes de loss avec et sans surapprentissage](../ressources/illustrations/jour_02/02_surapprentissage.png)

*Lorsque la validation remonte pendant que le train continue de baisser, le réseau mémorise davantage qu’il ne généralise.*


### Afficher puis interpréter les mesures

`history.history` est un dictionnaire : chaque nom, comme `"loss"`, désigne une liste de valeurs. Les deux courbes ci-dessous montrent le train et la validation, avec les mêmes axes pour pouvoir les comparer.

`evaluate` calcule la loss et l'accuracy sur le test sans modifier les poids. Le test mesure un modèle déjà choisi ; on ne l'utilise pas pour sélectionner une nouvelle architecture.


In [ ]:
plt.plot(history.history["loss"], label="train")  # Erreur des exemples utilisés pour apprendre.
plt.plot(history.history["val_loss"], label="validation")  # Erreur des exemples mis à part.
plt.xlabel("époque")
plt.ylabel("cross-entropy")
plt.title("Comparer l'apprentissage et la validation")
plt.legend()
plt.show()

test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)  # Aucune mise à jour.
print("Accuracy test :", round(test_accuracy, 3))


## Lire les sorties avant de calculer les classes

`predict` retourne un tableau avec une ligne par image et dix colonnes, une par chiffre. Une **ligne** rassemble les choix pour la même image. `axis=1` demande de travailler sur les colonnes, à l'intérieur de chaque ligne.

`argmax(axis=1)` renvoie la **position** du score le plus élevé : ici, cette position correspond à la classe. `max(axis=1)` renvoie sa **valeur**, appelée confiance du modèle. Exemple : `[0.06, 0.11, 0.70, 0.13]` donne la classe `2` et la confiance `0.70`.

Un score softmax décrit une préférence entre les classes proposées. Il ne garantit pas la vérité de la prédiction, ni le comportement sur une image très différente du train.


![La position maximum donne la classe et sa valeur donne la confiance](../ressources/illustrations/jour_02/04_argmax_max.svg)


In [ ]:
probabilities = model.predict(X_test, verbose=0)  # Une ligne de dix scores par image.
predictions = probabilities.argmax(axis=1)  # Indice du meilleur score de chaque ligne.
print("Forme des scores :", probabilities.shape)
print("Scores de la première image :", np.round(probabilities[0], 3))
print("Somme :", probabilities[0].sum(), "classe :", predictions[0], "réel :", y_test[0])


## Relier les nombres aux erreurs visibles

Une **matrice de confusion** compte les couples réel/prédit. Les lignes indiquent le chiffre réel ; les colonnes le chiffre prédit. Les cases de la diagonale comptent les réponses correctes. Une case hors diagonale révèle une confusion.

Exemple : la case ligne 3, colonne 8 compte les vrais `3` classés `8`. Une confusion mérite ensuite une inspection des images : certains chiffres sont-ils ambigus, flous ou peu représentés ?


In [ ]:
from sklearn.metrics import confusion_matrix  # Compte les couples réel/prédit.
import pandas as pd  # Ajoute des noms de lignes et de colonnes aux tableaux.

cm = confusion_matrix(y_test, predictions, labels=range(10))  # Dix lignes et dix colonnes.
display(pd.DataFrame(cm, index=[f"réel {i}" for i in range(10)],
                     columns=[f"prédit {i}" for i in range(10)]))


### Trouver les positions des erreurs

`predictions != y_test` compare chaque prédiction à sa réponse et produit des booléens : `True` signifie « différent ». `np.flatnonzero` retourne les indices où ce résultat vaut `True`. `[:5]` garde au maximum les cinq premières erreurs. Nous séparons leur sélection et leur affichage pour bien voir ces deux opérations.


## À vous de jouer — repérer puis afficher cinq erreurs

Complétez seulement `indices_erreurs` avec `np.flatnonzero(predictions != y_test)[:5]`. L'affichage est fourni. Pour une image, expliquez la confusion avec les pixels visibles avant de proposer un changement de modèle.


In [ ]:
indices_erreurs = None  # À compléter : positions des cinq premières erreurs.
if indices_erreurs is None:
    print("Sélectionnez les indices des erreurs.")
else:
    print("Indices sélectionnés :", indices_erreurs)


In [ ]:
if indices_erreurs is not None and len(indices_erreurs) > 0:
    # squeeze=False conserve une grille d'axes même si une seule erreur est affichée.
    fig, axes = plt.subplots(1, len(indices_erreurs), figsize=(10, 2.2), squeeze=False)
    for ax, index in zip(axes[0], indices_erreurs):
        ax.imshow(X_test[index], cmap="gray_r")  # L'image correspondant à cet indice.
        ax.set_title(f"réel {y_test[index]} / prédit {predictions[index]}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()
elif indices_erreurs is not None:
    print("Aucune erreur à afficher sur ce test.")


## Extension facultative — limiter la dépendance à un seul chemin

![Certaines sorties sont masquées pendant le train puis toutes sont utilisées en prédiction](../ressources/illustrations/jour_02/02_dropout_train_inference.png)

Le **Dropout** masque aléatoirement une fraction des sorties d'une couche pendant l'entraînement. `Dropout(0.20)` en masque environ 20 % à chaque passage. Ce choix change aléatoirement ; le réseau est encouragé à moins dépendre d'un seul chemin. Pendant `evaluate` et `predict`, le Dropout est désactivé.

L'arrêt anticipé et le Dropout jouent des rôles différents : le premier choisit quand arrêter, le second modifie ce que le réseau voit pendant le train. Aucun ne garantit une meilleure généralisation.

Pour essayer cette extension, ajoutez `tf.keras.layers.Dropout(0.20)` entre les deux couches Dense dans une **copie** du modèle, puis comparez la validation en gardant le même découpage. Le test final reste réservé au modèle choisi.


## Vérifier le fil avant le CNN

Dans l'ordre, expliquez : pourquoi normaliser ; pourquoi Flatten ; pourquoi dix sorties ; quel jeu sert à corriger les poids ; quel jeu déclenche l'arrêt ; ce que `argmax` retourne. Chaque réponse tient en une phrase. Une difficulté sur ces points indique l'étape à reprendre avant d'ajouter une convolution.
